# Exploratory Data Analysis

Loads the raw returns data saved by notebook 01 and explores:
- Return distributions per stock
- Correlation structure
- Rolling statistics
- Annualized risk / return per asset

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.visualization.plots import (
    plot_correlation_heatmap,
    plot_return_distributions,
    plot_risk_return_scatter,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

OverflowError: cannot convert longdouble infinity to integer

In [ ]:
log_returns = pd.read_csv('../data/raw/stock_log_returns.csv', index_col='Date')
log_returns.index = pd.to_datetime(log_returns.index)
returns     = pd.read_csv('../data/raw/stock_returns.csv', index_col='Date')
returns.index = pd.to_datetime(returns.index)

# Keep only numeric columns (drop any residual non-numeric)
log_returns = log_returns.select_dtypes(include='number')
returns     = returns.select_dtypes(include='number')

print(f'Shape: {returns.shape}')
print(returns.head(3))

In [ ]:
# Descriptive statistics
desc = returns.describe().T
desc['skew']     = returns.skew()
desc['kurtosis'] = returns.kurtosis()
desc

In [ ]:
# Return distribution histograms
fig = plot_return_distributions(returns, figsize=(16, 10))
plt.show()

In [ ]:
# Annualised risk / return per asset
periods = 252
ann_return = returns.mean() * periods
ann_vol    = returns.std(ddof=1) * np.sqrt(periods)

stats_df = pd.DataFrame({'Annualised Return': ann_return,
                         'Annualised Volatility': ann_vol})
stats_df['Sharpe (rf=2%)'] = (ann_return - 0.02) / ann_vol
print(stats_df.round(4))

In [ ]:
fig = plot_risk_return_scatter(
    mean_returns=ann_return.values,
    volatilities=ann_vol.values,
    labels=list(returns.columns),
)
plt.show()

In [ ]:
# Correlation heatmap
fig = plot_correlation_heatmap(returns.corr())
plt.show()

In [ ]:
# 30-day rolling volatility for each ticker
roll_vol = returns.rolling(30).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 5))
for col in roll_vol.columns:
    ax.plot(roll_vol.index, roll_vol[col], linewidth=0.9, label=col)
ax.set_title('30-Day Rolling Annualised Volatility')
ax.set_xlabel('Date')
ax.set_ylabel('Volatility')
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()